## Final Report: Voting Demographics
Team Members: Ashanti Hatchett, Srinath Rao Pathangae, Kumhyun Song

### Introduction

This project explores how voter demographics affect their participation in elections, using voter turnout data from the 2024 U.S. presidential election. The policy problem we address is unequal voter participation across demographic groups, which can undermine democratic representation.

Understanding which demographic groups are more or less likely to vote is important for policymakers and election officials seeking to increase overall voter turnout. If certain groups consistently participate at lower rates, their preferences may be underrepresented in electoral outcomes.

Using the results of our analysis, we aim to predict whether an individual voter is unlikely to vote. Such predictions could help inform targeted outreach strategies designed to encourage participation among voters who are less likely to vote. In this context, identifying individuals who ultimately do not vote is particularly important, as policy interventions would be directed toward these groups.

To make these predictions, we employ three models: logistic regression, random forest, and neural networks. We evaluate their performance using the F1 score, which is well-suited for our analysis due to the imbalanced nature of the dataset, as discussed in the Data section. We define not_voted as 1 and voted as 0, as our analysis focuses on identifying individuals who do not vote.

### Data

Our data source is the IPUMS CPS (Current Population Survey) November 2024 Voting Supplement (<https://cps.ipums.org/>). The dataset includes information on whether individuals voted in the 2024 U.S. presidential election, along with a wide range of demographic characteristics.
From the available variables, we selected 12 features based on their expected relationship with voting behavior:

- Sex (categorical): Sex is a standard demographic factor that may influence voting behavior. We included it because many previous studies suggest differences in political participation across sexes. In our dataset, males exhibit a slightly higher non-voting rate than females.

- Race (categorical, manipulated): Race is another important demographic variable affecting voting behavior. We grouped all smaller or mixed racial categories into an “other” category to reduce sparsity, as these categories contained relatively few observations. Our data show that individuals identified as American Indian or Eskimo have the highest non-voting rates.

![data1.png](plots/data1.png)

- Education (categorical, manipulated): Education level can influence political engagement and interest. Although more detailed subgroups were initially available, we used a simplified classification by grouping education into four main categories: below high school, high school graduate, college graduate, and master’s degree or higher. However, after data cleaning, no individuals remained in the below high school category. Individuals who attended college but did not graduate were grouped with high school graduates. Our results show that high school graduates have the highest non-voting rates.

- Employment Status (categorical, manipulated): Employment status may affect voting behavior through differences in time availability and political engagement. Although subgroups were initially available, we used the broader classification by consolidating them into four main categories: employed, unemployed, retired, and not in the labor force. Our analysis shows that unemployed and not-in-labor-force individuals tend to have higher non-voting rates.

![data2.png](plots/data2.png)

- Nativity (categorical, manipulated): Nativity indicates whether an individual is native-born or foreign-born. We simplified the variable by grouping individuals with at least one native-born parent as “native-born,” and others as “foreign-born.” This reflects differences in familiarity and engagement with U.S. political systems. Foreign-born individuals appear to have higher non-voting rates.

- Region (categorical, manipulated): Political behavior often varies by geographic region. We aggregated subregions into broader categories (Northeast, Midwest, South, and West). The Midwest and South show slightly higher non-voting rates, though differences are relatively modest.

![data3.png](plots/data3.png)

- Mobility Disability (categorical): Mobility limitations may restrict physical access to polling locations. Individuals with mobility-related disabilities show higher non-voting rates in our data.

- Metropolitan (categorical, manipulated): Access to polling locations may differ between metropolitan and non-metropolitan areas. We reduced this variable to a binary indicator (metropolitan vs. non-metropolitan). Individuals living in non-metropolitan areas tend to have slightly higher non-voting rates.

![data4.png](plots/data4.png)

- Marital Status (categorical, manipulated): Marital status may influence voting through social interaction and shared decision-making. We simplified this variable into two categories: with spouse and without spouse. Individuals without a spouse tend to have higher non-voting rates.

![data5.png](plots/data5.png)

- Age (numerical, manipulated): Age is a key predictor of voting behavior. In the dataset, individuals aged 80–84 are recorded as 80, and those aged 85 and above are recorded as 85, while all other ages are provided as exact values. To address this limitation, we grouped age into five-year intervals and used the midpoint of each interval. Younger individuals show significantly higher non-voting rates.

![data6.png](plots/data6.png)

- Number of Children (numerical): Having children may limit time available for voting. Our dataset indicates that households with more children tend to have higher non-voting rates.

![data7.png](plots/data7.png)

- Income Per Person (numerical, extracted): We created this feature by dividing family income by family size, as family income alone does not account for household size. Since family income was provided in ranges rather than exact values, we used the midpoint of each range as an estimate. This provides a more accurate measure of economic resources per individual. Lower income per person is associated with higher non-voting rates.

![data8.png](plots/data8.png)

We cleaned the dataset by removing observations with missing values. If more than one feature was missing (marked as “not in universe”), the entire row was dropped. After cleaning, the dataset contained 62,327 observations.

We then performed feature engineering and transformations as described above, including grouping categories, creating new variables, and simplifying complex variables. After that, we encoded categorical variables into numerical representations for model input using OneHotEncoder, and normalized numerical variables where appropriate using StandardScaler. Finally, we split the data into training (70%), validation (10%), and test (20%) sets for model development. 

One major limitation of the data is class imbalance, as there are significantly more individuals who voted than those who did not. This imbalance affects both model training and evaluation. We observed consistently high accuracy across different hyperparameter settings, making accuracy a less informative metric. Therefore, we used the F1 score for validation and evaluation, as it better captures the balance between precision and recall in imbalanced classification problems.

![data9.png](plots/data9.png)

Additionally, several variables are reported in categories rather than exact values. For example:

- Family income is provided in ranges rather than precise values, which affects the calculation of income per person (the extracted feature).
- Age is top-coded (80 for ages 80–84 and 85 for ages 85 and above).

These limitations reduce precision of our measurements.

If data were available, we would have liked to include variables related to transportation access, such as:

- Car ownership
- Access to public transportation

These factors could significantly affect physical access to polling locations. In the absence of such variables, we attempted to proxy access using metropolitan status and mobility disability.

### Model 1: Logistic Regression

We chose the logistic regression model to understand how our chosen features can explain someone's likelihood of voting on a probability level. We picked this to have more information beyond the binary classification, to hopefully better understand how different demographics contribute to a person's choice to vote.

We tuned three hyperparameters for this model:

- L1 vs. L2: We intended to use the L2 (Ridge) Regression as we expected multicollinearity and wanted to penalize the coefficients fairly. However, we still tested both in our validation to  see which regularizer resulted in a better model.
- C: A lower C provides stronger regularization, pushing coefficients closer to zero. So, we expected to have a quite small C, still we tested values from 0.0001 to 10.
- class weight (None vs. balanced): In the scikit learn's `LogisticRegression` class, class_weight='balanced' can help with imbalanced dataset, which may bias predictions toward our majority class of voted. Therefore, we expected a balanced class weight to perform better for this model.

As mentioned before, we chose to use the F1 score to capture the balance between precision and recall, as our data is imbalanced.

Our model with the highest mean F1 score in cross-validation had the following hyperparameters: 

- penalty = L1
- C = 0.0054
- class_weight = balanced

The model’s performance on the test set is as follows:

- F1 = 0.4967
- Recall = 0.6693
- Precision = 0.3949
- False Positive Rate = 0.3101
- False Negative Rate = 0.3307

![logistic_regression_hyperparams.png](plots/logistic_regression_hyperparams.png)

Based on the coefficients, the top 5 features that have the largest impact on one's odds of not voting are: `DIFFMOB_mobility_limitation`, `EDUC_hs_grad`, `EMPSTAT_not_in_labor_force`, `MARST_no_spouse`, and `NATIVITY_foreign_born`. These were expected based on our data exploration in which people with disabilities, high school education, no labor force participation, no spouse, and who were foreign born had higher rates of non-voters. Though many of these top features make sense, as things like disabilities present higher barriers to voting. However, those with no spouse having a higher odds of not voting was initially surprising -- it could be due to that being a younger population of people. Age is the third top feature contributing to a `voted` classification, suggesting that higher age increases one's odds of voting.

Further, `EDUC_master_higher` is the top feature that increases the odds of voting, which suggests that a person's education level is an important demographic in predicting their likelihood of voting.

![logistic_regression_coefficients.png](plots/logistic_regression_coefficients.png)

The confusion matrix for this model shows that, false negatives (957; predicting someone voted when they did not) are much lower compared to false positives (2968; predicting someone did not vote when they did). This is likely due to the imbalanced dataset, which contains more voters than non-voters. However, the false negative rate (0.33) is quite similar to the false positive rate (0.31), suggesting that the model is relatively balanced. We aimed to minimize false negatives to ensure we are identifying individuals who do not vote, for targeted voter engagement policy. Overall, this model does a decent job of that.

![logistic_regression_confusion_matrix.png](plots/logistic_regression_confusion_matrix.png)

### Model 2: Random Forest

Towards non-linear models, we considered using random forest and k-Nearest Neighbors (kNN). We chose the random forest model for the following reasons:

- It is relatively easy to interpret compared to other non-linear models, and it provides useful metrics such as feature importance, which help us understand the impact of each variable. 
- Additionally, the random forest handles non-linear relationships effectively and perform well with mixed types of features, making it well-suited for our dataset.

We decided not to use kNN because it can be biased toward the majority class in an imbalanced dataset, leading to poorer performance in identifying minority-class observations.

For the random forest implementation, the following hyperparameters were used:

- Number of Trees: Based on the OOB error and AUC plots, 500 trees were selected. The results indicate that performance improvements plateau after approximately 300 trees. Therefore, 500 trees were chosen as a conservative option. Since random forest is relatively robust to overfitting, increasing the number of trees can provide more stable performance on test data.

![random_forest_obb.png](plots/random_forest_obb.png)

![random_forest_performance.png](plots/random_forest_performance.png)

- Threshold for Majority Class: A threshold of 20% was selected, as it resulted in better predictive performance compared to a standard 50% threshold. This adjustment increased non-voter recall from 34% to 69%.

![random_forest_threshold.png](plots/random_forest_threshold.png)

- Class Weight: Since the dataset is imbalanced, the random forest would otherwise be biased toward predicting the majority class to achieve higher overall accuracy. However, this would lead to poor identification of non-voters, which contradicts the main objective of this analysis. So, class_weight parameter was set to "balanced" to assign greater penalties to errors in the minority class.

The model's performance on the test set is as follows:

- F1 = 0.4458  
- Recall = 0.6849  
- Precision = 0.3305  
- False Positive Rate = 0.4195  
- False Negative Rate = 0.3151

Permutation importance uses the validation dataset to shuffle the column of features to convert the feature into noise effectively, thereby measuring fall in performance as importance of the feature for model. Based on permutation importance, the two most important features are INCOME_PER_PERSON (0.0123), AGE (0.0065).

Our model includes one extracted feature, which appears among the most important features. This suggests that a well-constructed feature can capture meaningful information from the original variables.

The hyperparameter 'class_weight = balanced' did not fully address the issue of class imbalance. While it helps during tree construction by penalizing misclassification of the minority class more heavily, it does not directly affect the final prediction threshold. By setting the classification threshold to 0.20, the model better accounted for the imbalance in the data. Therefore, class imbalance was addressed in two stages: first through class weighting during training, and second through threshold adjustment during prediction.

Our random forest model used dummy variables to preserve consistency in model inputs across all models in this project. However, this self-imposed constraint could have diluted the importance of categorical features, since each dummy variable is treated as a separate feature rather than as a single categorical variable by random forest model.

### Model 3: Neural Networks

Neural networks are capable of learning complex, nonlinear relationships in data. Our goal is to predict whether an individual did not vote based on demographic characteristics, and neural networks are well-suited for capturing such patterns.

To train the model, we used the PyTorch library. The training process consists of the following steps: (1) for each training batch, perform a forward pass to compute predictions, (2) calculate the loss by comparing the predicted values (y_hat) with the true labels (y), (3) update model parameters using backpropagation, and (4) repeat this process over multiple epochs.

Because the dataset is imbalanced, we applied class weighting to give more importance to the minority class (`not_voted`). Specifically, we used the following code:

In [ ]:
self.pos_weight = torch.tensor([(self.torch_dataset.y_train == 0).sum()/(self.torch_dataset.y_train == 1).sum()]).float()

This increases the penalty for misclassifying individuals who did not vote, which aligns with the focus of our analysis.

We also observed that some models were unstable. Models with high variability in validation F1 scores (standard deviation greater than 0.1) were excluded to ensure stable performance:

In [ ]:
if f1_std > 0.1:
    result["f1"] = 0
    results.append(result)
    continue

Moreover, we used a stability-adjusted score (mean F1 minus standard deviation) to select the best model, ensuring both high performance and consistent behavior across training epochs.

In [ ]:
score = current_f1_mean - f1_std

In addition, models were excluded if their training loss failed to decrease or increased after several epochs, indicating poor convergence:

In [ ]:
elif epoch > 5 and avg_loss > initial_loss:
    return []

We tuned five key hyperparameters:

- Batch Size: To improve training efficiency compared to pure stochastic gradient descent (SGD), we used mini-batch training with batch sizes of 50, 100, and 200. For validation and testing, predictions were made without batching.

- Number of Hidden Layers: This determines the depth of the network. We tested models with 2, 5, and 10 hidden layers. While deeper networks can capture more complex patterns, they are also more prone to overfitting and instability.

- Hidden Dimension : This refers to the number of nodes in each hidden layer. Since the input dimension was 31 (after encoding), we tested 31 and 62. Larger dimensions increase model complexity.

- Learning Rate: This controls how quickly the model updates its parameters. A learning rate that is too high may cause divergence, while a rate that is too low may slow learning. We tested 0.01, 0.001, and 0.0001.

- Penalty Term: This represents the strength of L2 regularization. Strong regularization may lead to underfitting, while weak regularization may cause overfitting. We tested values of 0.01, 0.001, and 0.0001.

After running the hyperparameter search, we identified the best-performing model based on the validation F1 score:

- batch_size = 50
- hidden_layer = 2
- hidden_dim = 31
- learning_rate = 0.0001
- penalty = 0.001

As can be seen on the heatmap below, we found that models with a smaller number of hidden layers performed better, suggesting that simpler architectures were sufficient for this task and helped reduce overfitting.

![heatmap_hl_hd](plots/neural_networks_heatmap_hl_hd.png)

![f1_hidden_layers](plots/neural_networks_f1_hidden_layers.png)

The training curves show that performance quickly stabilizes and fluctuates slightly across epochs, suggesting limited improvement from additional training. Deeper architectures showed signs of overfitting, which further supports the choice of a simpler model. (Hidden layer with 10 is not shown in the plot, as these models were highly unstable and were excluded during training.)

Using the model with the best-performing hyperparameters, the performance on the test set is as follows:

- F1 = 0.5086
- Recall = 0.6842
- Precision = 0.4047
- False Positive Rate = 0.3042
- False Negative Rate = 0.3158

The false positive and false negative rates were similar in magnitude, indicating that the model achieved relatively balanced performance across classes.

### Model Evaluation Results

Based on the F1 scores, the worst-performing model was the random forest, while the best-performing model was the neural network.

The random forest model achieved a lower F1 score compared to the other models, which may be due to its sensitivity to class imbalance. Since our dataset is highly imbalanced, the model may have been biased toward predicting the majority class (voted), leading to lower precision in identifying non-voters despite adjustments such as class weighting and threshold tuning.

In contrast, the neural network achieved the highest F1 score. This may be due to its ability to learn complex, non-linear relationships in the data. Additionally, the neural network appears to have achieved a better balance between precision and recall, which is critical for optimizing the F1 score. Its flexibility likely allowed it to better capture underlying patterns associated with non-voting demographic characteristics, leading to improved performance.

### Surprises and Difficulties

One unexpected result was the relatively poor performance of the random forest model. Since random forests generally perform well in binary classification problems, we initially expected stronger results. However, in our case, the model struggled, likely due to the significant class imbalance, as discussed in the results section. Even after applying class weighting, we had to lower the classification threshold to around 20% to achieve reasonable recall for non-voters, which we found to be unexpectedly low.

Another surprising finding was that increasing the complexity of the neural network did not lead to better performance. Instead, a simpler architecture achieved the best results. A similar pattern was observed for logistic regression, which performed comparably to the other models despite its much simpler structure.

The most challenging aspect of the project was addressing class imbalance. Since non-voters accounted for only about 23% of the dataset, most models tended to favor the majority class. To mitigate this issue, we employed multiple strategies, including class weighting, threshold adjustment, and prioritizing the F1 score over accuracy for evaluation.

### Conclusion

Through this project, we've learned that the choice of scorer is very important to the interpretation of the model. Originally, we chose accuracy, but later learned that accuracy does not work well for imbalanced data. Then, we chose precision -- but we realized that we'd flipped our outcomes (voted = 1, rather than voted = 0), and precision didn't make sense as our main method of evaluation, since false positives were no longer our focus. Finally, we settled on the F1 score, since it is sensitive to how well the model handles the minority class, and it provides a balance between precision and recall.

From a machine learning perspective, we also learned that there is no single best model. Performance depends heavily on the data and the objective of the analysis. While more complex models such as neural networks can capture non-linear relationships, simpler models like logistic regression performed comparably in our case.

As for our research question, we learned that it's quite difficult to predict a person's likelihood of voting, as shown by the relatively low scores of all of our models (hovering around 0.5 for the F1 score). However, our recall scores were much higher than precision across the board, suggesting that our models did a decent job of correctly identifying non-voters, but also incorrectly classified a lot of voters as non-voters. This is likely due to the imbalance of our data, meaning that the models had more opportunities to learn what contributes to the odds of a person voting. Additionally, there may be many more factors that go into a choice not to vote that were not captured in our data such as access to reliable transportation and number of hours worked per day.

To further this research, we would aim to incorporate more detailed and relevant features at the individual level, such as transportation access, work hours, and family responsibilities. Expanding the feature set could help capture more of the underlying reasons why people do not vote.

If we had more time and unlimited resources, we would significantly expand the dataset to include a wider range of demographic, socioeconomic, and behavioral variables, potentially including geographic data. We would also explore additional features and more advanced modeling approaches to better understand not only prediction, but also the underlying drivers of voter participation.